# YOLOv8 training and fine-tuning

This notebook is a runnable training entry point using the recorded project settings. It expects the **correct Roboflow Version 4 YOLO dataset YAML** to be supplied locally; the full 8,481-image dataset is not in this repository. The training recipe is not a reconstruction of missing original run logs, so matching the reported metrics is not guaranteed.

Set `HELMET_DATA_YAML` to the Version 4 dataset config before training. The first run starts from the repository copy `model/coco/yolov8n.pt`; the second continues from that run with a data YAML that includes the difficult no-helmet examples. Do not point the fine-tuning run at the same data unless that is the desired experiment.

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "config" / "paths.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout.")

DATA_VALUE = os.environ.get("HELMET_DATA_YAML")
DATA_YAML = Path(DATA_VALUE).expanduser().resolve() if DATA_VALUE else None
if DATA_YAML is None or not DATA_YAML.is_file():
    raise FileNotFoundError("Set HELMET_DATA_YAML to the valid Version 4 data.yaml before training.")

IMAGE_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 40
RUNS_DIR = PROJECT_ROOT / "runs" / "helmet_violation"
print(f"Data config: {DATA_YAML}")
print(f"Run output: {RUNS_DIR}")

## Baseline training

This cell trains a baseline initialized from the standard YOLOv8 nano checkpoint. Ultralytics saves the output under the project `runs/` directory. The tracked `model/archive/helmet_best_old.pt` remains the archived baseline used by the application project comparison.

In [ ]:
baseline_weights = PROJECT_ROOT / "model" / "coco" / "yolov8n.pt"
if not baseline_weights.is_file():
    raise FileNotFoundError(f"Baseline weights not found: {baseline_weights}")
baseline_model = YOLO(str(baseline_weights))
baseline_run = baseline_model.train(
    data=str(DATA_YAML),
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    epochs=EPOCHS,
    project=str(RUNS_DIR),
    name="baseline_reproduction",
)
baseline_best = Path(baseline_run.save_dir) / "weights" / "best.pt"
print(f"Baseline checkpoint: {baseline_best}")

## Fine-tuning with difficult no-helmet cases

Prepare a separate Version 4 YAML whose training split includes the difficult no-helmet examples intended for this experiment. The file and its images are external to this repository. Fine-tuning starts from the baseline run above and saves to a separate run directory; it does not replace the app checkpoint.

In [ ]:
FINETUNE_DATA_VALUE = os.environ.get("HELMET_FINETUNE_DATA_YAML", str(DATA_YAML))
FINETUNE_DATA_YAML = Path(FINETUNE_DATA_VALUE).expanduser().resolve()
if not FINETUNE_DATA_YAML.is_file():
    raise FileNotFoundError(f"Fine-tuning data YAML not found: {FINETUNE_DATA_YAML}")
if not baseline_best.is_file():
    raise FileNotFoundError(f"Run the baseline cell first; checkpoint not found: {baseline_best}")

finetuned_model = YOLO(str(baseline_best))
finetuned_run = finetuned_model.train(
    data=str(FINETUNE_DATA_YAML),
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    epochs=EPOCHS,
    project=str(RUNS_DIR),
    name="finetuned_difficult_no_helmet",
)
print(f"Fine-tuned checkpoint: {Path(finetuned_run.save_dir) / 'weights' / 'best.pt'}")